# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step workflow for loading, exploring, and processing the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is accessed via a Croissant schema URL and adheres to the MLCommons Croissant format.

In [ ]:
# If not already installed, install the mlcroissant library
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Explore the dataset's available record sets, their fields, and their `@id` identifiers.

> All entities are referenced by their `@id`. Use these IDs for programmatic access via Croissant-compatible tools.

In [ ]:
# List available record sets, fields, and columns, displaying their @id and human-readable name
print("Available record sets in the dataset:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    print(f"  name: {rs.get('name', 'N/A')}")
    if 'field' in rs:
        print("  Fields:")
        for field in rs['field']:
            fname = field.get('name', 'N/A')
            print(f"    - Field @id: {field['@id']}, name: {fname}")
            if 'column' in field:
                print("      Columns:")
                for col in field['column']:
                    cname = col.get('name', 'N/A')
                    print(f"        - Column @id: {col['@id']}, name: {cname}")
    print()

## 3. Data Extraction
Load data from each record set into DataFrames for further analysis.

> Use exact `@id` values gathered above when specifying record sets and fields.

In [ ]:
# Gather all record set @id values
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print("  No records found for this RecordSet.")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Loaded {len(df)} records with columns:")
    print(f"  {df.columns.tolist()}")

# Preview the first DataFrame (if available)
if dataframes:
    sample_record_set_id = list(dataframes.keys())[0]
    print(f"\nSample data from RecordSet @id: {sample_record_set_id}")
    display(dataframes[sample_record_set_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping for one selected record set.

For demonstration, select a numeric field (e.g., "age") and a grouping field (e.g., "sex") from the first available record set.

In [ ]:
# Identify the first available dataframe for EDA
if dataframes:
    main_rs_id = sample_record_set_id
    df = dataframes[main_rs_id]
    print(f"Using records from RecordSet @id: {main_rs_id}")
    print("Columns available:")
    print(df.columns.tolist())
    
    # Try to guess a numeric and a groupable field, using @id (here we look for 'age' or similar for demo purposes)
    numeric_candidates = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or 'year' in col.lower()]
    numeric_field_id = numeric_candidates[0] if numeric_candidates else df.columns[0]
    
    group_field_candidates = [col for col in df.columns if 'sex' in col.lower() or 'msi' in col.lower() or 'group' in col.lower() or 'anatomical' in col.lower()]
    group_field_id = group_field_candidates[0] if group_field_candidates else None
    
    print(f"Selected numeric field for analysis: {numeric_field_id}")
    if group_field_id:
        print(f"Selected group field: {group_field_id}")
    else:
        print("No group field identified.")
    
    # Convert numeric field to float if possible
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean()
    print(f"Filtering records where {numeric_field_id} > {threshold:.2f}")
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered {filtered_df.shape[0]} records (of {df.shape[0]}).")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Grouped mean
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize distributions or relationships in the data. We'll provide a histogram and, if possible, a group comparison.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=10)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    
    # Grouped boxplot if a grouping field exists
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("Visualization skipped: no data or fields available.")

## 6. Conclusion
- Using `mlcroissant`, we loaded, explored, and visualized a FAIR-compliant clinical colorectal cancer dataset using only Croissant `@id` fields.
- We demonstrated accessing record sets, fields, and data, as well as filtering and normalization steps for exploratory data analysis.
- These steps offer a reproducible workflow for downstream analytics, modelling, or domain-specific hypotheses.